# 模型推理 - 使用 QLoRA 微调后的 ChatGLM-6B
# 使用epoch=3，automode-dataset(fixed)：20240118_163659.csv

In [2]:
import torch
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig

# 模型ID或本地路径
model_name_or_path = 'THUDM/chatglm3-6b'

In [3]:
_compute_dtype_map = {
    'fp32': torch.float32,
    'fp16': torch.float16,
    'bf16': torch.bfloat16
}

# QLoRA 量化配置
q_config = BitsAndBytesConfig(load_in_4bit=True,
                              bnb_4bit_quant_type='nf4',
                              bnb_4bit_use_double_quant=True,
                              bnb_4bit_compute_dtype=_compute_dtype_map['bf16'])

# 加载量化后模型(与微调的 revision 保持一致）
base_model = AutoModel.from_pretrained(model_name_or_path,
                                      quantization_config=q_config,
                                      device_map='auto',
                                      trust_remote_code=True,
                                      revision='b098244')

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [4]:
base_model.requires_grad_(False)
base_model.eval()

ChatGLMForConditionalGeneration(
  (transformer): ChatGLMModel(
    (embedding): Embedding(
      (word_embeddings): Embedding(65024, 4096)
    )
    (rotary_pos_emb): RotaryEmbedding()
    (encoder): GLMTransformer(
      (layers): ModuleList(
        (0-27): 28 x GLMBlock(
          (input_layernorm): RMSNorm()
          (self_attention): SelfAttention(
            (query_key_value): Linear4bit(in_features=4096, out_features=4608, bias=True)
            (core_attention): CoreAttention(
              (attention_dropout): Dropout(p=0.0, inplace=False)
            )
            (dense): Linear4bit(in_features=4096, out_features=4096, bias=False)
          )
          (post_attention_layernorm): RMSNorm()
          (mlp): MLP(
            (dense_h_to_4h): Linear4bit(in_features=4096, out_features=27392, bias=False)
            (dense_4h_to_h): Linear4bit(in_features=13696, out_features=4096, bias=False)
          )
        )
      )
      (final_layernorm): RMSNorm()
    )
    (output_la

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path,
                                          trust_remote_code=True,
                                          revision='b098244')

## 使用原始 ChatGLM3-6B 模型

In [6]:
input_text = "解释下坤卦是什么？"

In [7]:
response, history = base_model.chat(tokenizer, query=input_text)

In [8]:
print(response)

坤卦是《易经》中的一个卦象，它的卦辞为“地载万物，承载承载，含养万物。”坤卦代表大地，象征着承载、承载和滋养。在《易经》中，坤卦具有顺从、承载、滋养的特点，代表着一种强烈的责任感和使命感。

坤卦的六爻分别有不同的含义：

1. 初六：阴爻，表示柔顺、温柔、初生的生命力。
2. 六二：阴爻，表示柔顺、温柔、初生的生命力，但已有一定的发展。
3. 六三：阴爻，具有阳刚之氣，象征着柔顺中有刚毅，温柔中有力量。
4. 六四：阳爻，表示刚毅、坚定、有决断。
5. 六五：阳爻，具有柔顺、承载的品质，象征着领导者的风范。
6. 六六：阳爻，表示刚毅、坚定、有决断，具有领导者的风范。

坤卦在《易经》中具有很高的地位，它与乾卦（象征天、刚强、刚毅）相互配合，形成乾坤之象，象征着宇宙间阴阳二元对立和谐的关系。通过研究坤卦，我们可以了解到天地间万物的运行规律，从而指导我们在生活中的行为和决策。


#### 询问一个64卦相关问题（应该不在 ChatGLM3-6B 预训练数据中）

In [9]:
response, history = base_model.chat(tokenizer, query="周易中的讼卦是什么？", history=history)
print(response)

讼卦是《易经》中的一个卦象，它的卦辞为“天作之合，人工之成，可爱可敬。”讼卦象征诉讼、争端和纷争。在周易中，讼卦表示通过沟通、协调和合作来解决矛盾和争端，具有和谐、公正、公平的特点。

讼卦的六爻分别有不同的含义：

1. 初六：阴爻，表示柔顺、温柔、初生的生命力。
2. 六二：阴爻，表示柔顺、温柔、初生的生命力，但已有一定的发展。
3. 六三：阴爻，具有阳刚之氣，象征着柔顺中有刚毅，温柔中有力量。
4. 六四：阳爻，表示刚毅、坚定、有决断。
5. 六五：阳爻，具有柔顺、承载的品质，象征着领导者的风范。
6. 六六：阳爻，表示刚毅、坚定、有决断，具有领导者的风范。

讼卦在《易经》中具有很高的地位，它与乾卦（象征天、刚强、刚毅）相互配合，形成乾坤之象，象征着天地间万物的运行规律。通过研究讼卦，我们可以了解到天地间万物的运行规律，从而指导我们在生活中的行为和决策。


## 使用微调后的 ChatGLM3-6B

### 加载 QLoRA Adapter(Epoch=3, automade-dataset(fixed)) - 请根据训练时间戳修改 timestamp 

In [11]:
from peft import PeftModel, PeftConfig

epochs = 3
# timestamp = "20240118_164514"
timestamp = "20240314_111734"

peft_model_path = f"models/{model_name_or_path}-epoch{epochs}-{timestamp}"

config = PeftConfig.from_pretrained(peft_model_path)
qlora_model = PeftModel.from_pretrained(base_model, peft_model_path)
training_tag=f"ChatGLM3-6B(Epoch=3, automade-dataset(fixed))-{timestamp}"

In [12]:
def compare_chatglm_results(query, base_model, qlora_model, training_tag):
    base_response, base_history = base_model.chat(tokenizer, query)

    inputs = tokenizer(query, return_tensors="pt").to(0)
    ft_out = qlora_model.generate(**inputs, max_new_tokens=512)
    ft_response = tokenizer.decode(ft_out[0], skip_special_tokens=True)
    
    print(f"问题：{query}\n\n原始输出：\n{base_response}\n\n\n微调后（{training_tag}）：\n{ft_response}")
    return base_response, ft_response

### 微调前后效果对比

In [13]:
base_response, ft_response = compare_chatglm_results("解释下坤卦是什么？", base_model, qlora_model, training_tag)

问题：解释下坤卦是什么？

原始输出：
坤卦，是周易中的一卦，由两个坤卦叠加而成，代表大地的顺从和承载。在这个卦中，预示着大吉大利，预言了雌马牵动的吉兆，君子出行会先迷失，后来找到主人，有利的方向是西南，不利的方向是东北。总体上，这是一个吉利的卦象。《象辞》中讲到，大地形势平和，君子观卦以厚德载物。坤卦的解释中提到，坤卦代表柔顺和地气舒展之象，主张妥善安排，等待时机，宜顺从运势以制定大事。在传统解卦中，坤卦代表谨慎行事，灵活适应，依循正道获得吉利。在事业、经商、婚姻和决策等方面，坤卦均主张顺从自然规律，勿急进，谋求长远利益。


微调后（ChatGLM3-6B(Epoch=3, automade-dataset(fixed))-20240314_111734）：
[gMASK]sop 解释下坤卦是什么？ 坤卦，是周易中的一卦，由两个坤卦叠加而成，代表大地的顺从和承载。在这个卦中，预示着大吉大利，预言了雌马牵动的吉兆，君子出行会先迷失，后来找到主人，有利的方向是西南，不利的方向是东北。总体上，这是一个吉利的卦象。《象辞》中讲到，大地形势平和，君子观卦以厚德载物。坤卦的解释中提到，坤卦代表柔顺和地气舒展之象，主张妥善安排，等待时机，宜顺从运势以制定大事。在传统解卦中，坤卦代表谨慎行事，灵活适应，依循正道获得吉利。在事业、经商、婚姻和决策等方面，坤卦均主张顺从自然规律，勿急进，谋求长远利益。


In [14]:
base_response, ft_response = compare_chatglm_results("周易中的讼卦是什么", base_model, qlora_model, training_tag)

问题：周易中的讼卦是什么

原始输出：
在周易中，讼卦是一个充满警示的卦象。它由上卦乾（天）和下卦坎（水）组成，代表着天与水背道而驰，形成争讼的局面。虽然事情开始时有利可图，但必须警惕戒惧，因为中间虽然吉利，但最终会带来凶险。对于涉及大川，涉水渡河的行动不利。因此，君子观此卦象，应当慎之又慎，杜绝争讼之事，并在谋事之初谨慎行事。讼卦的核心哲学是要避免争讼，退而让人，求得化解，安于正理，方可避免意外之灾。在事业上，务必避免介入诉讼纠纷的争执之中，与其这样，不如退而让人。即使最终获胜，也难免得失不均。经商方面，要坚持公正、公平、互利的原则，避免冲突，这样会有好结果。而对于求名、婚恋和决策，也都需要慎重行事，避免盲目追求，退让让人，可助事业、婚姻和决策的发展。


微调后（ChatGLM3-6B(Epoch=3, automade-dataset(fixed))-20240314_111734）：
[gMASK]sop 周易中的讼卦是什么样子 讼卦是一个充满警示的卦象。它由上卦乾（天）和下卦坎（水）组成，代表着天与水背道而驰，形成争讼的局面。虽然事情开始时有利可图，但必须警惕戒惧，因为中间虽然吉利，但最终会带来凶险。对于涉及大川，涉水渡河的行动不利。因此，君子观此卦象，应当慎之又慎，杜绝争讼之事，并在谋事之初谨慎行事。讼卦的核心哲学是要避免争讼，退而让人，求得化解，安于正理，方可避免意外之灾。在事业上，务必避免介入诉讼纠纷的争执之中，与其这样，不如退而让人。即使最终获胜，也难免得失不均。经商方面，要坚持公正、公平、互利的原则，避免冲突，这样会有好结果。而对于求名、婚恋和决策，也都需要慎重行事，避免盲目追求，退让让人，可助事业、婚姻和决策的发展。


In [15]:
base_response, ft_response = compare_chatglm_results("师卦是什么？", base_model, qlora_model, training_tag)

问题：师卦是什么？

原始输出：
{'name': '占卜结果可以参考下述示例:', 'content': '\n(注:以下为占卜结果的描述,仅供参考)\n\n您所询问师卦的结果为“升”卦。这个结果意味着增加、上升、进步和成功。在事业、工作、学业或人际关系方面都有积极的含义。不过,需要注意到是在特定时间、地点和条件下才有效。\n\n对于事业方面,意味着将开始一个新的职位或项目,或者在现有职位或项目中有所升迁。需要努力学习和工作,以获得他人的认可和信任,从而获得更多的机会和晋升。\n\n对于工作方面,意味着将开始一个新的工作或转行到新的行业。需要积极拓展人脉,获得更多的机会和晋升,同时要努力工作以证明自己的能力。\n\n对于学业方面,意味着将开始一个新的学习阶段或考试。需要努力学习,积极寻求帮助,以及认真对待考试,以获得好成绩。\n\n对于人际关系方面,意味着将开始一个新的社交圈子或面临重要的社交场合。需要积极与他人交流,展示自己的个性和魅力,以及尊重他人的需求和利益。'}


微调后（ChatGLM3-6B(Epoch=3, automade-dataset(fixed))-20240314_111734）：
[gMASK]sop 师卦是什么？ 师卦是一个由坎卦（水）和坤卦（地）相叠而成的异卦。在周易中，它代表着军队的力量和军情的总指挥，预示着吉祥无灾。象辞中描述了地中有水的情景，寓意着君子应当像大地一样容纳和畜养大众。师卦的解释强调选择德高望重的长者来统率军队，才能获得吉祥无咎。另外，师卦也象征着困难重重，需要包容别人、艰苦努力，及时行事，严于律已。在事业、经商、求名、婚恋等方面的决策中，都需要警惕潜在敌人，小心谨慎，合作与决断兼顾，方能成功。
